# Azure AI Foundry Agent with Microsoft Learn MCP Tool
This notebook demonstrates how to use the Microsoft Learn MCP server with Azure AI Foundry Agents.
No authentication is required for the MCP server.

## Step 1: Imports

In [ ]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool
from openai.types.responses.response_input_param import McpApprovalResponse, ResponseInputParam


## Step 2: Configuration
Update `PROJECT_ENDPOINT` with your Azure AI Foundry project endpoint.
Format: `https://<resource_name>.services.ai.azure.com/api/projects/<project_name>`

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Locate notebook directory
notebook_path = Path(globals().get("__vsc_ipynb_file__", ".")).resolve()
current_dir = notebook_path.parent

# Search upward for .env
env_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / ".env"
    if candidate.exists():
        env_path = candidate
        break

if not env_path:
    raise FileNotFoundError("❌ .env file not found in project directory hierarchy")

# Load env file
load_dotenv(env_path)

print(f"✅ Loaded .env from: {env_path}")
print()

# Required variables
PROJECT_ENDPOINT = os.environ["PROJECT_ENDPOINT"]
MODEL_NAME = os.environ["MODEL_NAME"]
AGENT_NAME = os.environ["AGENT_NAME"]

# MCP config
MCP_SERVER_URL = os.environ["MCP_SERVER_URL"]
MCP_SERVER_LABEL = os.environ["MCP_SERVER_LABEL"]
MCP_REQUIRE_APPROVAL = os.environ["MCP_REQUIRE_APPROVAL"]

# Optional
MCP_CONNECTION_ID = os.environ.get("MCP_CONNECTION_ID", "")

print(f"PROJECT_ENDPOINT     : {PROJECT_ENDPOINT}")
print(f"MODEL_NAME           : {MODEL_NAME}")
print(f"AGENT_NAME           : {AGENT_NAME}")
print(f"MCP_SERVER_URL       : {MCP_SERVER_URL}")
print(f"MCP_SERVER_LABEL     : {MCP_SERVER_LABEL}")
print(f"MCP_REQUIRE_APPROVAL : {MCP_REQUIRE_APPROVAL}")
print(f"MCP_CONNECTION_ID    : {'(set)' if MCP_CONNECTION_ID else '(not set – public server)'}")

## Step 3: Create the AI Project Client

In [ ]:
project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

openai_client = project.get_openai_client()

print("✅ Clients created successfully")


## Step 4: Define the MCP Tool
The Microsoft Learn MCP server exposes a `microsoft_docs_search` tool.
Because this is a read-only public server, we set `require_approval='never'`.
No `project_connection_id` is needed since there is no auth.

In [ ]:
# Build keyword args – only include project_connection_id when set
mcp_kwargs = dict(
    server_label=MCP_SERVER_LABEL,
    server_url=MCP_SERVER_URL,
    require_approval=MCP_REQUIRE_APPROVAL,
)
if MCP_CONNECTION_ID:
    mcp_kwargs["project_connection_id"] = MCP_CONNECTION_ID

mcp_tool = MCPTool(**mcp_kwargs)

print(f"✅ MCP Tool configured: {MCP_SERVER_URL}")
if MCP_CONNECTION_ID:
    print(f"   Connection ID : {MCP_CONNECTION_ID}")


## Step 5: Create the Agent

In [ ]:
agent = project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL_NAME,
        instructions=(
            "You are a helpful assistant that answers questions about Microsoft technologies. "
            "Always use the microsoft_docs_search MCP tool to search the official Microsoft Learn "
            "documentation before answering. Ground your responses in the docs you retrieve."
        ),
        tools=[mcp_tool],
    ),
)

print(f"✅ Agent created")
print(f"   ID      : {agent.id}")
print(f"   Name    : {agent.name}")
print(f"   Version : {agent.version}")


## Step 6: Create a Conversation

In [ ]:
conversation = openai_client.conversations.create()

print(f"✅ Conversation created: {conversation.id}")

## Step 7: Send a Question and Get a Response
This question is deliberately specific to Microsoft docs so the MCP tool will be triggered.

In [ ]:
USER_QUESTION = "What is Azure AI Agent Service and how does MCP tool calling work with it?"

print(f"🙋 User: {USER_QUESTION}\n")
print("⏳ Sending request to agent...\n")

response = openai_client.responses.create(
    conversation=conversation.id,
    input=USER_QUESTION,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)

# -------------------------------------------------------
# Debug: show every output item returned
# -------------------------------------------------------
print("📦 Output items received:")
for i, item in enumerate(response.output):
    print(f"   [{i}] type={item.type}")

print()

## Step 8: Handle MCP Approval Requests (if any)
Since we set `require_approval='never'`, this block is usually skipped.
It is included here as a fallback in case the server overrides the setting.

In [ ]:
input_list: ResponseInputParam = []
approval_needed = False

for item in response.output:
    if item.type == "mcp_approval_request" and item.id:
        approval_needed = True

        print("🔐 MCP Approval Requested")
        print(f"   Server    : {item.server_label}")
        print(f"   Tool      : {getattr(item, 'name', '<unknown>')}")
        print(f"   Arguments : {json.dumps(getattr(item, 'arguments', None), indent=4, default=str)}\n")

        # Auto-approve since microsoft_learn is safe and read-only.
        # Change to False to block the call.
        should_approve = True
        print(f"   ✅ Auto-approved: {should_approve}")

        input_list.append(
            McpApprovalResponse(
                type="mcp_approval_response",
                approve=should_approve,
                approval_request_id=item.id,
            )
        )

# Send the approval response back only if an approval was actually needed
if approval_needed and input_list:
    print("\n⏳ Sending approval response back to agent...")
    response = openai_client.responses.create(
        input=input_list,
        previous_response_id=response.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    )
    print("✅ Approval response sent\n")
else:
    print("ℹ️  No approval required (tool ran automatically or was not triggered)\n")

## Step 9: Print the Final Response

In [ ]:
print("🤖 Agent Response:")
print("-" * 60)
print(response.output_text)
print("-" * 60)

## Step 10: Follow-up Question (Multi-turn)
Because we reuse the same conversation ID, the agent remembers context.

In [ ]:
FOLLOWUP_QUESTION = "Can you give me a Python code example of creating an MCP tool with the Azure AI SDK?"

print(f"🙋 Follow-up: {FOLLOWUP_QUESTION}\n")
print("⏳ Sending follow-up...\n")

followup_response = openai_client.responses.create(
    conversation=conversation.id,
    input=FOLLOWUP_QUESTION,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)

print("🤖 Agent Response:")
print("-" * 60)
print(followup_response.output_text)
print("-" * 60)

## Step 11: Clean Up
Always delete the agent version when done to free up resources.

In [ ]:
project.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
print(f"🗑️  Agent '{agent.name}' version {agent.version} deleted successfully")